In [27]:
%%bash

cat << 'EOF' > /content/config.sh
#!/bin/bash

export SCENE="table_clem"

export ROOTDIR="/content/${SCENE}"
export NUM_FRAMES=100
export NUM_JOBS=2
export GIT_BRANCH="dev"

export PREPROCESS_PROFILE="hloc-lightblue"
export GSPLAT_PROFILE="balanced"
export GSPLAT_PROFILE="quality_plus"
export GSPLAT_PROFILE="experiment"
export GSPLAT_PROFILE="quality"

export BASENAME="${SCENE}_${GSPLAT_PROFILE}"

if [ "$SCENE" = "soldat" ]; then

  export VIDEOSOURCE="gsplat/input/Video_20260515_1755_16_064.MOV"
  export VIDEO_START=10
  export VIDEO_END=88

elif [ "$SCENE" = "statue" ]; then

  export VIDEOSOURCE="gsplat/input/IMG_4812.MOV"
  #export VIDEO_START=0
  #export VIDEO_END=100

elif [ "$SCENE" = "table_clem" ]; then

  export VIDEOSOURCE="gsplat/input/IMG_4811.MOV"
  #export VIDEO_START=0
  #export VIDEO_END=100
fi
EOF

cat /content/config.sh

#!/bin/bash

export SCENE="table_clem"

export ROOTDIR="/content/${SCENE}"
export NUM_FRAMES=100
export NUM_JOBS=2
export GIT_BRANCH="dev"

export PREPROCESS_PROFILE="hloc-lightblue"
export GSPLAT_PROFILE="balanced"
export GSPLAT_PROFILE="quality_plus"
export GSPLAT_PROFILE="experiment"
export GSPLAT_PROFILE="quality"

export BASENAME="${SCENE}_${GSPLAT_PROFILE}"

if [ "$SCENE" = "soldat" ]; then

  export VIDEOSOURCE="gsplat/input/Video_20260515_1755_16_064.MOV"
  export VIDEO_START=10
  export VIDEO_END=88

elif [ "$SCENE" = "statue" ]; then

  export VIDEOSOURCE="gsplat/input/IMG_4812.MOV"
  #export VIDEO_START=0
  #export VIDEO_END=100

elif [ "$SCENE" = "table_clem" ]; then

  export VIDEOSOURCE="gsplat/input/IMG_4811.MOV"
  #export VIDEO_START=0
  #export VIDEO_END=100
fi


In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
%%bash
set -e
source /content/config.sh

mkdir -p "$ROOTDIR"

if [ -n "$VIDEOSOURCE" ]; then
  SRC_VIDEO="/content/drive/MyDrive/$VIDEOSOURCE"

  if [ ! -f "$SRC_VIDEO" ]; then
    echo "❌ Video not found: $SRC_VIDEO"
  else
    echo "🎬 Copying video: $SRC_VIDEO"
    cp -f "$SRC_VIDEO" "$ROOTDIR/video.mp4"
  fi
fi

🎬 Copying video: /content/drive/MyDrive/gsplat/input/IMG_4811.MOV


In [30]:
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
!bash Miniforge3-Linux-x86_64.sh -b -p /usr/local/miniforge

# activer conda pour cette session notebook
import os
os.environ["PATH"] = "/usr/local/miniforge/bin:" + os.environ["PATH"]

!conda --version

ERROR: File or directory already exists: '/usr/local/miniforge'
If you want to update an existing installation, use the -u option.
conda 26.3.2


In [31]:
!git clone https://github.com/NicoIGN/video_to_ply.git
%cd video_to_ply

Cloning into 'video_to_ply'...
remote: Enumerating objects: 2384, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 2384 (delta 103), reused 122 (delta 58), pack-reused 2207 (from 1)
Receiving objects: 100% (2384/2384), 792.86 KiB | 3.15 MiB/s, done.
Resolving deltas: 100% (1510/1510), done.
/content/video_to_ply/video_to_ply/video_to_ply


In [32]:
%%bash
cd /content/video_to_ply
source /content/config.sh
git stash save && git checkout $GIT_BRANCH && git pull

No local changes to save
Your branch is up to date with 'origin/dev'.
Updating cc2f7e5..a471ef4
Fast-forward
 profiles/gsplat/experiment.sh | 2 +-
 profiles/gsplat/quality.sh    | 6 +++---
 2 files changed, 4 insertions(+), 4 deletions(-)


Already on 'dev'
From https://github.com/NicoIGN/video_to_ply
   cc2f7e5..a471ef4  dev        -> origin/dev


In [33]:
!source /usr/local/miniforge/etc/profile.d/conda.sh && \
mamba env list | grep -q gsplat && \
cd /content/video_to_ply/ && \
mamba run -n gsplat mamba env update -n gsplat -f environment/conda_colab.yml --prune -y || \
mamba env create -n gsplat -f environment/conda_colab.yml -y

[+] 0.0s
[+] 0.1s
pytorch/linux-64 (..  ⣾  
pytorch/noarch (ch..  ⣾  
nvidia/linux-64 (c..  ⣾  
nvidia/noarch (che..  ⣾  [+] 0.2s
nvidia/linux-64 (c..  ⣾  ⚠ Shard Index for pytorch/linux-64 not available, falling back to flat repodata
Using Flat Repodata for pytorch/linux-64                                                  ✔ Done (0.0 sec)
⚠ Shard Index for pytorch/noarch not available, falling back to flat repodata
Using Flat Repodata for pytorch/noarch                                                    ✔ Done (0.0 sec)
⚠ Shard Index for nvidia/linux-64 not available, falling back to flat repodata
Using Flat Repodata for nvidia/linux-64                                                   ✔ Done (0.0 sec)
⚠ Shard Index for nvidia/noarch not available, falling back to flat repodata
Using Flat Repodata for nvidia/noarch                                                     ✔ Done (0.0 sec)
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ D

In [34]:
%%bash
source /usr/local/miniforge/etc/profile.d/conda.sh

SITE_PACKAGES=$(mamba run -n gsplat python -c "import site; print(site.getsitepackages()[0])")
TARGET="$SITE_PACKAGES/SuperGluePretrainedNetwork"

if [ ! -d "$TARGET" ]; then
    git clone --depth 1 \
        https://github.com/magicleap/SuperGluePretrainedNetwork.git \
        "$TARGET"
else
    echo "SuperGluePretrainedNetwork already installed."
fi

Cloning into '/usr/local/miniforge/envs/gsplat/lib/python3.10/site-packages/SuperGluePretrainedNetwork'...


In [ ]:
!RUN=1; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
 (source /usr/local/miniforge/etc/profile.d/conda.sh && \
source /content/config.sh && \
cd /content/video_to_ply/ && \
mamba run -n gsplat bash run.sh  \
--root "$ROOTDIR" \
--name "$BASENAME" \
--video $ROOTDIR/video.mp4  \
--skip-conda \
--preprocess-profile "$PREPROCESS_PROFILE" \
--gsplat-profile "$GSPLAT_PROFILE" \
--num-frames $NUM_FRAMES \
--max-jobs $NUM_JOBS \
--no-proxy)

🚫 Proxy disabled (NO_PROXY=true)
👉 using profile: preprocess/hloc-lightblue
👉 using profile: gsplat/quality
⏩ Skipping conda setup (--skip-conda enabled)
✅ Using python: Python 3.10.20 
🚀 GPU model OK: splatfacto
📦 ROOT: /content/table_clem
🎬 Extracting 100 sharp frames → /content/table_clem/input/images
🎬 Video: /content/table_clem/input/video.mov
📁 Output (IMAGE_DIR): /content/table_clem/ori/images
⏩ VIDEO_START: 10s
⏹️ VIDEO_END: 88s
⚙️ Mode: SMART selection (100)
📐 Oversampling: 300 frames
⏱️ Effective duration: 78.0 s
⏱️ Interval: 0.26 s
frame=  300 fps=2.9 q=-0.0 Lsize=N/A time=00:01:18.00 bitrate=N/A speed=0.75x    
🔎 Adaptive filtering...
📊 After blur filter: 210
📐 Adaptive diff threshold: 9.116241078317902
📊 Candidates: 202
✅ Selected frames: 100
🔎 Checking image geometry consistency...
✅ All images have consistent size: (1920, 1080)
✅ Final frames: 100

⏱️  INPUT PREPARATION completed in 02m 10s



🧭 Preprocessing hloc-lightblue through NerfStudio...
📝 Process log: /content/t

In [ ]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
 (source /usr/local/miniforge/etc/profile.d/conda.sh && \
source /content/config.sh && \
cd /content/video_to_ply/ && \
mamba run -n gsplat bash run.sh  \
--root "$ROOTDIR" \
--name "$BASENAME" \
--video $ROOTDIR/video.mp4  \
--skip-conda \
--preprocess-profile "$PREPROCESS_PROFILE" \
--gsplat-profile "$GSPLAT_PROFILE" \
--fps $FPS \
--no-proxy)

In [ ]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
 (source /usr/local/miniforge/etc/profile.d/conda.sh && \
source /content/config.sh && \
cd /content/video_to_ply/ && \
mamba run -n gsplat bash run.sh  \
--root "$ROOTDIR" \
--name "$BASENAME" \
--video $ROOTDIR/video.mp4  \
--skip-conda \
--skip-training \
--preprocess-profile "$PREPROCESS_PROFILE" \
--gsplat-profile "$GSPLAT_PROFILE" \
--num-frames $NUM_FRAMES \
--no-proxy)

In [ ]:
from google.colab import files
import os
import subprocess

# =========================
# LOAD CONFIG.SH VARIABLES
# =========================
result = subprocess.run(
    "source /content/config.sh && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

# =========================
# CONFIG
# =========================
rootdir = os.environ.get("ROOTDIR", "/content/work")
export_dir = os.path.join(rootdir, "exports")
basename = os.environ.get("BASENAME", "")

if not basename:
    print("❌ BASENAME is not set")
    raise SystemExit(1)


# =========================
# EXPORT ORIGINAL PLY
# =========================
base_ply = os.path.join(export_dir, f"{basename}.ply")
if os.path.exists(base_ply):
    print(f"⬇️ Downloading original PLY: {os.path.basename(base_ply)}")
    files.download(base_ply)
else:
    print(f"⚠️ Original PLY not found: {base_ply}")


In [ ]:
from google.colab import files
import os
import subprocess
import os

# =========================
# LOAD CONFIG.SH VARIABLES
# =========================
result = subprocess.run(
    "source /content/config.sh && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

# =========================
# CONFIG
# =========================
rootdir = os.environ.get("ROOTDIR", "/content/work")
export_dir = os.path.join(rootdir, "exports")
basename = os.environ.get("BASENAME", "")

if not basename:
    print("❌ BASENAME is not set")
    raise SystemExit(1)


# =========================
# EXPORT COLMAP DATA
# =========================
base_zip = os.path.join(export_dir, f"colmap_{basename}.zip")
if os.path.exists(base_zip):
    print(f"⬇️ Downloading COLMAP data: {os.path.basename(base_zip)}")
    files.download(base_zip)
else:
    print(f"⚠️ COLMAP data not found: {base_zip}")

# =========================
# EXPORT LATEST RUN ZIP
# =========================

zip_files = [
    os.path.join(export_dir, f)
    for f in os.listdir(export_dir)
    if (
        f.endswith(".zip")
        and not f.startswith("colmap_")
    )
]

if not zip_files:
    print(f"⚠️ No zip archives found in: {export_dir}")
else:
    latest_zip = max(zip_files, key=os.path.getmtime)

    print(f"⬇️ Downloading latest run archive: {os.path.basename(latest_zip)}")
    files.download(latest_zip)